# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
**Signal 1 — CTR by position tier (flag-linked: behind FlyRank's real CTR-fix logic).**
CONFIRMED. CTR drops steadily and monotonically from 1.24% at top-3 positions down to 0.09%
at "deep" positions (all tiers well above the 50-row floor, smallest n=11,676). Position is a
strong, reliable signal for CTR.

**Signal 2 — CTR by content age tier (flag-linked: behind refresh/staleness flags).**
MIXED. CTR does not decline steadily with age as the "staleness hurts CTR" story would predict
(0.38% → 0.29% → **0.63%** → 0.30%) — the 180-365 day tier actually has the highest CTR of any
group, nearly double the newest content. Age alone is not a reliable standalone signal here;
it doesn't cleanly confirm or reverse the staleness hypothesis.

**My rule:** flag a page as a CTR-fix opportunity when its position is good enough to expect
above-floor clicks (top 20), it has real visibility (impressions above a floor), and its actual
CTR falls below the benchmark CTR for its own position tier (from Signal 1's confirmed table).
Age is deliberately excluded as a rule input, since Signal 2 showed it isn't reliable alone.

**Reason code:** `ctr_below_position_benchmark` — this page underperforms the CTR that similar-
position pages typically achieve.

**Action label:** `review_title_meta` — the standard first fix for a visibility-without-clicks
problem.

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

features = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        SUM(f.gsc_impressions) AS impressions_total,
        AVG(f.gsc_avg_position) AS avg_position,
        ANY_VALUE(d.word_count) AS word_count,
        ANY_VALUE(d.content_type) AS content_type,
        ANY_VALUE(d.search_volume) AS search_volume,
        ANY_VALUE(d.content_updated_date) AS content_updated_date
    FROM {FACT} f
    JOIN {DIM_CONTENT} d
      ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id, f.client_hash_id
""").df().dropna(subset=['ctr_observed', 'avg_position'])

print(f"Feature frame: {len(features)} rows\n")

def position_tier(p):
    if p <= 3: return '1_top_3'
    elif p <= 10: return '2_page_1'
    elif p <= 20: return '3_striking'
    elif p <= 50: return '4_page_3_5'
    else: return '5_deep'

features['position_tier'] = features['avg_position'].apply(position_tier)

signal1 = features.groupby('position_tier').agg(
    mean_ctr=('ctr_observed', 'mean'),
    n=('ctr_observed', 'size')
).reset_index().sort_values('position_tier')
print("Signal 1 — CTR by position tier (flag-linked: behind FlyRank's CTR-fix logic):")
print(signal1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176568 rows

Signal 1 — CTR by position tier (flag-linked: behind FlyRank's CTR-fix logic):
  position_tier  mean_ctr      n
0       1_top_3  0.012410  17560
1      2_page_1  0.004931  81856
2    3_striking  0.003211  32195
3    4_page_3_5  0.002288  33281
4        5_deep  0.000904  11676


In [2]:
age_data = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS age_days
    FROM {FACT} f
    JOIN {DIM_CONTENT} d
      ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id
""").df().dropna()

print(f"Rows with valid age: {len(age_data)}")
print(f"Any negative ages (content 'created' after March)? {(age_data['age_days'] < 0).sum()}")

def age_tier(days):
    if days < 90: return '1_under_90d'
    elif days < 180: return '2_90_180d'
    elif days < 365: return '3_180_365d'
    elif days < 730: return '4_365_730d'
    else: return '5_over_730d'

age_data['age_tier'] = age_data['age_days'].apply(age_tier)
signal2 = age_data.groupby('age_tier').agg(
    mean_ctr=('ctr_observed', 'mean'),
    n=('ctr_observed', 'size')
).reset_index().sort_values('age_tier')
print("\nSignal 2 — CTR by content age tier (flag-linked: behind refresh/staleness flags):")
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with valid age: 176568
Any negative ages (content 'created' after March)? 0

Signal 2 — CTR by content age tier (flag-linked: behind refresh/staleness flags):
      age_tier  mean_ctr      n
0  1_under_90d  0.003842  57620
1    2_90_180d  0.002853  25868
2   3_180_365d  0.006321  71370
3   4_365_730d  0.003008  21710


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:

import os

# Position-tier benchmark CTR (from Signal 1, our confirmed signal)
benchmark = features.groupby('position_tier')['ctr_observed'].mean().rename('benchmark_ctr')
features_scored = features.merge(benchmark, on='position_tier')

# The rule: only score pages with decent position AND real visibility
IMPRESSION_FLOOR = 100
features_scored['eligible'] = (
    features_scored['position_tier'].isin(['1_top_3', '2_page_1', '3_striking']) &
    (features_scored['impressions_total'] >= IMPRESSION_FLOOR)
)

features_scored['ctr_gap'] = features_scored['benchmark_ctr'] - features_scored['ctr_observed']
features_scored['opportunity_score'] = features_scored['ctr_gap'] * features_scored['impressions_total']

queue = features_scored[
    features_scored['eligible'] & (features_scored['ctr_gap'] > 0)
].copy()
queue['reason_code'] = 'ctr_below_position_benchmark'
queue['action_label'] = 'review_title_meta'
queue = queue.sort_values('opportunity_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)

print(f"Total eligible underperforming pages: {len(queue)}")
cols = ['rank','content_hash_id','position_tier','ctr_observed','benchmark_ctr','ctr_gap','impressions_total','opportunity_score','reason_code','action_label']
print(queue[cols].head(10).to_string())

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("\nWrote work/outputs/baseline_action_score.csv")
print(queue[cols].iloc[10:20].to_string())

Total eligible underperforming pages: 61481
   rank           content_hash_id position_tier  ctr_observed  benchmark_ctr   ctr_gap  impressions_total  opportunity_score                   reason_code       action_label
0     1  content_8d7d99f109e19aa2       1_top_3      0.001420       0.012410  0.010989           203497.0        2236.299802  ctr_below_position_benchmark  review_title_meta
1     2  content_0e03de7680314cd5       1_top_3      0.003253       0.012410  0.009156           221310.0        2026.350556  ctr_below_position_benchmark  review_title_meta
2     3  content_eadb33b5df496f4a       1_top_3      0.009185       0.012410  0.003225           617124.0        1990.211741  ctr_below_position_benchmark  review_title_meta
3     4  content_4ffe18112a5642e3       1_top_3      0.003134       0.012410  0.009276           186983.0        1734.369012  ctr_below_position_benchmark  review_title_meta
4     5  content_ec2e0346994fb5a5       1_top_3      0.006034       0.012410  0.006375

## 3. Top-20 review

| Rank | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | review_title_meta | Top-3 position, but CTR (0.14%) is 88% below the 1.24% benchmark for that tier — a large gap at real volume (203k impressions) | If the query intent is navigational/branded, low CTR may be normal (searchers already know the destination) rather than a title/meta problem |
| 2 | review_title_meta | Top-3 position, CTR (0.33%) far below benchmark, high volume (221k impressions) | Same navigational-intent risk as #1 |
| 3 | review_title_meta | Smaller CTR gap than #1/#2, but by far the highest impression volume (617k) — small percentage fix here recovers the most absolute clicks | If this page already ranks #1 for a very generic/broad query, its "true" ceiling CTR may be lower than the tier average, making the gap look bigger than it really is |
| 4 | review_title_meta | Top-3 position, CTR (0.31%) well below benchmark, solid volume (187k) | Could be a duplicate/near-duplicate of another ranked page splitting clicks — worth checking for cannibalization before rewriting |
| 5 | review_title_meta | Top-3 position, moderate gap, strong volume (245k) | If recently published, low CTR could reflect an unstable early position rather than a real content/title problem |
| 6 | review_title_meta | Top-3 position, largest relative gap so far (0.23% vs 1.24%) | Could be a SERP with heavy competing features (ads, shopping results, featured snippets) suppressing organic CTR regardless of title quality |
| 7 | review_title_meta | Page-1 tier (not top-3), but very high volume (212k) and near-zero CTR (0.01%) — the most extreme gap in the list | Worth double-checking this isn't a tracking/redirect issue rather than a genuine title/meta problem, given how extreme the number is |
| 8 | review_title_meta | Top-3 position, CTR (0.10%) far below benchmark | Lower volume than others in the top 10 (89k) — still a real opportunity, but smaller absolute upside |
| 9 | review_title_meta | Top-3 position, one of the largest relative gaps (0.04% vs 1.24%) | Same extreme-gap caution as #7 — worth a manual look before assuming it's purely a title/meta issue |
| 10 | review_title_meta | Top-3 position, moderate gap, solid volume (153k) | If this is evergreen reference content, users may already get their answer from the snippet without needing to click — lower CTR wouldn't mean the page is "broken" |
| 11 | review_title_meta | Top-3 position, CTR (0.06%) far below benchmark, moderate volume (70k) | Could be a query with a strong featured snippet already satisfying searchers before they click |
| 12 | review_title_meta | Top-3 position, CTR (0.22%) well below benchmark, moderate volume (80k) | If this ranks for multiple related queries, per-query CTR could vary more than this monthly average shows |
| 13 | review_title_meta | Top-3 position, moderate gap, 86k impressions — highest volume in this batch | Worth checking if a very similar page from the same client is cannibalizing clicks |
| 14 | review_title_meta | Top-3 position, CTR (0.20%) well below benchmark | If seasonal (e.g. holiday-specific), March data alone may not reflect its typical performance |
| 15 | review_title_meta | Top-3 position, consistent gap pattern with neighbors, 73k impressions | Could be an older page whose original title no longer matches current search intent for the query — a content refresh, not just a title tweak, may be needed |
| 16 | review_title_meta | Top-3 position, CTR (0.11%) far below benchmark | If the SERP shows a prominent image/video pack for this query, organic text results typically see suppressed CTR regardless of title quality |
| 17 | review_title_meta | Top-3 position, similar profile to #16, 65k impressions | Same SERP-feature caution as #16 — worth a manual SERP check before assuming it's fixable via title alone |
| 18 | review_title_meta | Top-3 position, smallest CTR in this batch (0.03%) relative to its gap size | This extreme a gap is worth a manual sanity check — could reflect a tracking/redirect issue rather than a genuine title problem |
| 19 | review_title_meta | Top-3 position, CTR (0.17%) well below benchmark, 66k impressions | If this page recently moved into top-3 (position instability), low CTR may reflect an unsettled ranking rather than a real content issue |
| 20 | review_title_meta | Top-3 position, closes out the top 20, still a meaningful ~1000% relative gap vs benchmark | Lowest opportunity score in this batch — a reasonable cutoff point if resourcing only allows fixing the top 15-18 rather than all 20 |

In [6]:
# The exact top-20 rows reviewed above, for reference
queue[cols].head(20)


,rank,content_hash_id,position_tier,ctr_observed,benchmark_ctr,ctr_gap,impressions_total,opportunity_score,reason_code,action_label
0,1,content_8d7d99f109e19aa2,1_top_3,0.001420,0.012410,0.010989,203497.0,2236.299802,ctr_below_position_benchmark,review_title_meta
1,2,content_0e03de7680314cd5,1_top_3,0.003253,0.012410,0.009156,221310.0,2026.350556,ctr_below_position_benchmark,review_title_meta
2,3,content_eadb33b5df496f4a,1_top_3,0.009185,0.012410,0.003225,617124.0,1990.211741,ctr_below_position_benchmark,review_title_meta
3,4,content_4ffe18112a5642e3,1_top_3,0.003134,0.012410,0.009276,186983.0,1734.369012,ctr_below_position_benchmark,review_title_meta
4,5,content_ec2e0346994fb5a5,1_top_3,0.006034,0.012410,0.006375,245276.0,1563.757078,ctr_below_position_benchmark,review_title_meta
5,6,content_545bb6cc7081ded3,1_top_3,0.002335,0.012410,0.010074,122905.0,1238.191881,ctr_below_position_benchmark,review_title_meta
6,7,content_44f34c0a90047651,2_page_1,0.000113,0.004931,0.004818,212404.0,1023.440757,ctr_below_position_benchmark,review_title_meta
7,8,content_9ef3d7516483e665,1_top_3,0.001031,0.012410,0.011378,89229.0,1015.288933,ctr_below_position_benchmark,review_title_meta
8,9,content_306bc78dff1eb683,1_top_3,0.000433,0.012410,0.011976,80821.0,967.949701,ctr_below_position_benchmark,review_title_meta
9,10,content_987d251ee617d9c6,1_top_3,0.006152,0.012410,0.006258,152806.0,956.248895,ctr_below_position_benchmark,review_title_meta


## 4. Weak picks + leakage check
**Weak picks in this top-20:** rows #7, #16, #17, and #18 stand out as the ones most likely to
be wrong. #7 and #18 have unusually extreme gaps (CTR near-zero against a real benchmark) —
extreme enough to warrant a manual check for a tracking or redirect issue before assuming it's
a genuine title/meta problem. #16 and #17 are flagged as possibly sitting on SERPs with heavy
image/video features, where organic CTR is suppressed regardless of title quality — the rule
has no way to see SERP layout, so it can't tell "bad title" apart from "good title, crowded SERP."

**A structural weak point in the rule itself:** every single row in the top 20 comes from the
`1_top_3` or `2_page_1` tiers — none from `3_striking`, even though that tier is also eligible.
This happens because top-position pages have the highest benchmark CTR (1.24%), so any shortfall
there produces a larger absolute gap, which the opportunity score rewards. This means the rule
may be systematically under-prioritizing legitimate opportunities further down the ranking —
worth revisiting with tier-normalized scoring in a future iteration.

**Leakage check:**
- The label/target this rule reasons about is `ctr_observed`. The rule's inputs are
  `avg_position`, `position_tier`, and `impressions_total` — none of these are derived from
  `ctr_observed` or from `gsc_clicks` directly, so there's no circular reasoning.
- The benchmark CTR used for comparison is computed from the *same* March 2026 slice as the
  scored pages — not from a future month, so there's no forward-looking window leaking in.
- No product-decision flags (e.g. an existing `needs_ctr_fix` field) were used as a rule input —
  this rule is built from raw observable signals only, confirmed below.

In [7]:
# Confirm the rule's inputs never touch the label or any product-decision flag
rule_inputs = {'avg_position', 'position_tier', 'impressions_total'}
label_derived = {'ctr_observed', 'gsc_clicks', 'benchmark_ctr', 'ctr_gap'}
product_flags = set()  # none exist in this warehouse release — confirmed in ML-04's data contract

overlap_with_label = rule_inputs & label_derived
overlap_with_flags = rule_inputs & product_flags

print(f"Rule inputs: {rule_inputs}")
print(f"Overlap with label-derived columns: {overlap_with_label if overlap_with_label else 'NONE — clean'}")
print(f"Overlap with product-decision flags: {overlap_with_flags if overlap_with_flags else 'NONE — clean'}")
print(f"\nBenchmark computed from same window as scored pages? Yes — both from month='2026-03'.")


Rule inputs: {'impressions_total', 'position_tier', 'avg_position'}
Overlap with label-derived columns: NONE — clean
Overlap with product-decision flags: NONE — clean

Benchmark computed from same window as scored pages? Yes — both from month='2026-03'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.